In [54]:
import pandas as pd
from joblib import load
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    roc_auc_score
)
import sys
sys.path.insert(0, "../../")
from utils.ploting_figures import MakePlots
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE

In [55]:
def metrics(predict, y, dataset, div, predict_proba=None):
    acc_value = accuracy_score(y_pred=predict, y_true=y) 
    recall_value = recall_score(y_pred=predict, y_true=y, average='weighted')
    precision_value = precision_score(y_pred=predict, y_true=y, average='weighted') 
    f1_value = f1_score(y_pred=predict, y_true=y, average='weighted')
    mcc_value = matthews_corrcoef(y_pred=predict, y_true=y)
    cm = confusion_matrix(y_pred=predict, y_true=y)
    cm_dict = pd.DataFrame(cm).to_dict()

    roc_auc_value = None
    if predict_proba is not None:
        try:
            roc_auc_value = roc_auc_score(y, predict_proba, multi_class='ovr', average='weighted')
        except Exception as e:
            print(f"Error computing ROC AUC: {e}")
            roc_auc_value = None

    df_metrics = pd.DataFrame([[dataset, "Random_Forest", div, acc_value, recall_value, precision_value, f1_value, mcc_value, roc_auc_value, cm_dict]],
                              columns=["dataset", "model", "sampling", "acc", "recall", "precision", "f1", "mcc", "roc_auc", "conf_matrix"])

    return df_metrics

In [56]:
model_original = load('../../models/RandomForest_Grid_42_Original_best.joblib')
model_undersampling = load('../../models/RandomForest_Grid_42_Under_best.joblib')
model_oversampling = load('../../models/RandomForest_Grid_42_Over_best.joblib')

In [57]:
def test(X_test, y_test, model_original, model_undersampling, model_oversampling, name):
    y_pred_original = model_original.predict(X_test)
    y_proba_original = model_original.predict_proba(X_test)
    y_pred_under = model_undersampling.predict(X_test)
    y_proba_under = model_undersampling.predict_proba(X_test)
    y_pred_over = model_oversampling.predict(X_test)
    y_proba_over = model_oversampling.predict_proba(X_test)

    original_metrics= metrics(y_pred_original, y_test, name,'Original', predict_proba=y_proba_original)
    undesampling_metrics = metrics(y_pred_under, y_test, name,'Undersampling', predict_proba=y_proba_under)
    oversampling_metrics = metrics(y_pred_over, y_test, name,'Oversampling', predict_proba=y_proba_over)
    return pd.concat([original_metrics, undesampling_metrics, oversampling_metrics], ignore_index=True)

In [58]:
name_independent = "data_independent"
name_excluded= "embedding_excluded_homology_protT5"
#df_data = pd.read_csv(f"../../data/{name_data}.csv")
independent= pd.read_csv(f"../../data/numerical_rep/indep/{name_independent}.csv")
excluded = pd.read_csv(f"../../data/numerical_rep/indep/{name_excluded}.csv")
excluded = excluded.drop(columns=['experimental_characteristics'])

In [59]:
X_test_indep=independent.drop(columns=['target'])
y_test_indep=independent['target']
X_test_excl=excluded.drop(columns=['target'])
y_test_excl=excluded['target']

In [60]:
metrics_indep = test(X_test_indep, y_test_indep, model_original, model_undersampling, model_oversampling, 'Test_independent')
metrics_excl = test(X_test_excl, y_test_excl, model_original, model_undersampling, model_oversampling, 'Test_excluded')

In [61]:
metrics_df= pd.concat([metrics_indep, metrics_excl], ignore_index=True)
metrics_df.to_csv(f"../../metrics/RandomForest_indep_excl_data.csv", index=False)